#### Exercise 5: Interactive AI Prototyping with Gradio

In [ ]:
pip install gradio transformers torch accelerate

##### Import libraries

In [ ]:
import gradio as gr
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

##### Load the pretrained AI model

In [ ]:
# Check available device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# Lightweight Hugging Face model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

print("Model loaded successfully!")

##### Create the chatbot function

In [ ]:

def chatbot_response(user_message, conversation_history):

    # Initialize the message list with system instructions
    chat_messages = [{
        "role": "system",
        "content": "You are a helpful AI assistant. Give clear and concise answers."
    }]

    # Add previous user and assistant messages
    for entry in conversation_history:
        role = entry["role"]
        content = entry["content"]

        if role in ("user", "assistant") and isinstance(content, str):
            chat_messages.append({
                "role": role,
                "content": content
            })

    # Append the latest user input
    chat_messages.append({
        "role": "user",
        "content": user_message
    })

    # Convert messages into the model's chat format
    formatted_text = tokenizer.apply_chat_template(
        chat_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize the formatted conversation
    model_inputs = tokenizer(
        formatted_text,
        return_tensors="pt"
    ).to(device)

    # Generate the assistant's reply
    with torch.inference_mode():
        generated_output = model.generate(
            **model_inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract and decode newly generated tokens
    new_tokens = generated_output[0][model_inputs["input_ids"].shape[1]:]

    reply = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return reply

##### Create and launch Gradio chatbot interface

In [ ]:
# Stop any existing Gradio applications
gr.close_all()

# Set up the chatbot interface
demo = gr.ChatInterface(
    fn=chatbot,

    title="Interactive AI Chatbot — Prototype",

    description=(
        "Welcome to the interactive AI assistant! "
        "This prototype showcases conversational AI capabilities "
        "built with Gradio and Hugging Face. "
        "Type your query or choose an example below to get started."
    ),

    examples=[
        "Can you explain Artificial Intelligence?",
        "Describe Machine Learning with a practical example.",
        "How does NIC contribute to Digital India?",
        "How are AI and Deep Learning different?"
    ]
)

# Start the Gradio application
demo.launch(
    share=False,
    debug=True
)